# dots.ocr — Kaggle GPU Server

**Trước khi chạy:** Bật GPU bằng cách vào Settings (góc phải) → Accelerator → **GPU T4 x1**


## Cell 1 — Kiểm tra GPU + cài packages

In [ ]:
import subprocess
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)

!pip install "vllm>=0.11.0" pyngrok -q
import vllm
print(f'✅ vLLM {vllm.__version__} installed')

## Cell 2 — Clone repo + tải model (~4GB, 5-10 phút)

In [ ]:
import os

# Clone repo
if not os.path.exists('/kaggle/working/dots.ocr'):
    !git clone https://github.com/hoanggiangppe-tech/dots.ocr.git /kaggle/working/dots.ocr

%cd /kaggle/working/dots.ocr
!git checkout claude/setup-local-repo-4LmIb

# Download model
if not os.path.exists('/kaggle/working/dots.ocr/weights/DotsMOCR'):
    !python3 tools/download_model.py
    print('✅ Model downloaded')
else:
    print('✅ Model đã có sẵn')

## Cell 3 — Patch model code (chạy 1 lần)

In [ ]:
import os

def patch_file(file_path):
    with open(file_path, 'r') as f:
        lines = f.readlines()
    new_lines = []
    modified = False
    i = 0
    while i < len(lines):
        line = lines[i]
        stripped = line.lstrip()
        indent = line[:len(line) - len(stripped)]
        if any(f'Auto{t}.register' in line for t in ['Config','Model','Processor','Tokenizer']):
            if i == 0 or 'try:' not in lines[i-1]:
                new_lines += [f'{indent}try:\n', f'{indent}    {stripped}',
                              f'{indent}except (ValueError, AssertionError):\n',
                              f'{indent}    pass  # already registered\n']
                modified = True
                i += 1
                continue
        new_lines.append(line)
        i += 1
    if modified:
        with open(file_path, 'w') as f:
            f.writelines(new_lines)
    return modified

patched = []
for root, _, files in os.walk('./weights/DotsMOCR'):
    for fname in files:
        if fname.endswith('.py'):
            fpath = os.path.join(root, fname)
            if patch_file(fpath):
                patched.append(fname)
                print(f'✅ Patched: {fname}')

print(f'Done — patched {len(patched)} file(s)')

## Cell 4 — Start server + tạo ngrok URL

In [ ]:
import subprocess, time, requests, os
from pyngrok import ngrok, conf

# ⚠️ Dán ngrok token của bạn vào đây
# Lấy tại: https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_TOKEN = ""  # <-- dán token vào đây

if not NGROK_TOKEN:
    print('⚠️ Chưa có token! Dán token vào NGROK_TOKEN = "..." rồi chạy lại')
else:
    conf.get_default().auth_token = NGROK_TOKEN

    # Fix FlashInfer lcuda error: tạo symlink libcuda.so
    r1 = subprocess.run(['ln', '-sf', '/usr/local/cuda/lib64/stubs/libcuda.so', '/usr/local/lib/libcuda.so'], capture_output=True)
    subprocess.run(['ldconfig'], capture_output=True)
    print('✅ CUDA symlink fix applied')

    # Kill process cũ
    subprocess.run(['pkill', '-f', 'vllm'], capture_output=True)
    ngrok.kill()
    time.sleep(3)

    # Tạo tunnel
    tunnel = ngrok.connect(8000, bind_tls=True)
    public_url = tunnel.public_url
    print('\n' + '='*55)
    print(f'🌐 PUBLIC URL: {public_url}')
    print('='*55)
    print('👆 Dán URL này vào ô Server URL trong UI local!\n')

    # Env: tắt FlashInfer JIT để tránh lỗi lcuda nếu symlink không đủ
    env = os.environ.copy()
    env['VLLM_FLASHINFER_DISABLE'] = '1'

    # Start vLLM
    log_file = open('/tmp/vllm.log', 'w')
    proc = subprocess.Popen([
        'vllm', 'serve', './weights/DotsMOCR',
        '--tensor-parallel-size', '1',
        '--gpu-memory-utilization', '0.95',
        '--max-model-len', '32768',
        '--chat-template-content-format', 'string',
        '--served-model-name', 'model',
        '--trust-remote-code',
        '--port', '8000',
    ], stdout=log_file, stderr=log_file, env=env)

    print('⏳ Đang khởi động (3-5 phút lần đầu)...')
    for i in range(120):
        time.sleep(5)
        if proc.poll() is not None:
            log_file.flush()
            print(f'❌ Server crash! Exit code: {proc.poll()}')
            with open('/tmp/vllm.log') as f:
                lines = f.readlines()
            errors = [l for l in lines if any(k in l for k in ['ERROR','ValueError','RuntimeError','failed','OOM'])]
            print(''.join(errors[-15:]))
            break
        try:
            if requests.get('http://localhost:8000/v1/models', timeout=3).status_code == 200:
                print(f'\n✅ Server sẵn sàng sau {(i+1)*5}s!')
                print(f'🔗 URL: {public_url}')
                break
        except:
            if (i+1) % 6 == 0:
                print(f'   [{(i+1)*5}s] Đang khởi động...')
    else:
        print('⚠️ Timeout.')

## Cell 5 — Keep-alive (chạy sau khi server sẵn sàng)

In [ ]:
# Kaggle tự động giữ session khi cell đang chạy
# Cell này giữ server sống bằng cách ping định kỳ
import time, requests

print('🔄 Keep-alive đang chạy (Ctrl+C hoặc Stop để dừng)...')
count = 0
while True:
    time.sleep(60)
    count += 1
    try:
        r = requests.get('http://localhost:8000/v1/models', timeout=5)
        if count % 5 == 0:  # in mỗi 5 phút
            print(f'   [{count} phút] Server OK ✅')
    except:
        print(f'   [{count} phút] ⚠️ Server không phản hồi!')